# BirdCLEF+ 2025 - Fast CPU inference: MAIN noisy student

**CPU only, internet off.** This single notebook is switched by `MODEL` (`b0` = our team's baseline v2, `a1_v2s` = A1) and `BACKEND` (`torch`, `onnx`, `openvino_fp16`). It follows the same protocol as `infer-v2`: twelve 5 s windows per soundscape, with the same `row_id` and columns.

We only changed things that affect speed. 4 spawned worker processes handle whole files with 1 thread each, each file gets one `melspectrogram` call on a (windows x samples) array, and a file's windows go through the model as one batch. The mel is still computed per window (each window keeps its own centre padding), so the values match the per-window loop of the existing notebooks up to float32 round-off. dB scaling and `to_input` are still applied per window and come from each model's `cfg.json`.

With `RUN_ON = "auto"` the notebook predicts `test_soundscapes` when it has files (a scored submission) and otherwise times `TIMING_N_FILES` files from `train_soundscapes`. The exports are built here from the attached `model.pth`. The notebook stops if a GPU is visible, because the timings are only valid on CPU.


In [ ]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[v] = "1"                # set before numpy loads; each worker uses one core
import time
T_NOTEBOOK = time.time()

import torch
CUDA_AVAILABLE = torch.cuda.is_available()
DEVICE = "cpu"
print("torch.cuda.is_available():", CUDA_AVAILABLE)
print("DEVICE:", DEVICE)
if CUDA_AVAILABLE:
    raise RuntimeError("A GPU is visible: timings are only valid on CPU. Run this notebook without an accelerator.")

# ---- configuration: edit these, or override with the M3_* environment variables ----
MODEL = os.environ.get("M3_MODEL", "a1_v2s")                    # "b0" or "a1_v2s"
BACKEND = os.environ.get("M3_BACKEND", "torch")             # "torch", "onnx" or "openvino_fp16"
RUN_ON = os.environ.get("M3_RUN_ON", "auto")                # "auto", "train_soundscapes" or "test_soundscapes"
TIMING_N_FILES = int(os.environ.get("M3_N_FILES", "700"))   # train_soundscapes runs only
NUM_WORKERS = int(os.environ.get("M3_INFER_WORKERS", "4"))  # 0 = no Pool
BUDGET_MIN = 75
ONNX_LOGIT_TOL = 1e-3     # float32 round-off between runtimes only
FP16_LOGIT_TOL = 0.25     # weights are rounded to half precision, so a real but small shift is expected

MODELS = {"b0": dict(kernel="sc4000-birdclef-m1-train-v2", arch="efficientnet_b0"),
          "a1_v2s": dict(kernel="sc4000-birdclef-m3-ns-student-main", arch="tf_efficientnetv2_s")}
if MODEL not in MODELS: raise ValueError(f"unknown MODEL {MODEL}")
if BACKEND not in ("torch", "onnx", "openvino_fp16"): raise ValueError(f"unknown BACKEND {BACKEND}")
if RUN_ON not in ("auto", "train_soundscapes", "test_soundscapes"): raise ValueError(f"unknown RUN_ON {RUN_ON}")

import glob, importlib, importlib.metadata, importlib.util, json, multiprocessing as mp, subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd

WORK = Path(os.environ.get("M3_WORK_DIR", "."))
WORK.mkdir(parents=True, exist_ok=True)

def find_data_dir():
    if os.environ.get("M3_DATA_DIR"):       # local tests only
        return Path(os.environ["M3_DATA_DIR"])
    hits = glob.glob("/kaggle/input/**/birdclef-2025/train.csv", recursive=True) \
        or glob.glob("/kaggle/input/*/train.csv")
    return Path(hits[0]).parent if hits else Path("../../data")

def find_model_dir(kernel):
    """Both model kernels are attached and both hold a model.pth, so match on the kernel name."""
    if os.environ.get("M3_MODEL_DIR"):      # local tests only
        return Path(os.environ["M3_MODEL_DIR"])
    hits = [h for h in glob.glob("/kaggle/input/**/model.pth", recursive=True) if kernel in h]
    if len(hits) != 1:
        raise RuntimeError(f"expected one model.pth under kernel {kernel}, found {hits}")
    return Path(hits[0]).parent

DATA, MODEL_DIR = find_data_dir(), find_model_dir(MODELS[MODEL]["kernel"])
CFG = json.load(open(MODEL_DIR / "cfg.json"))
species = json.load(open(MODEL_DIR / "species.json"))
CFG.setdefault("norm", "v2"); CFG.setdefault("norm_top_db", 80.0)   # M1 cfg.json has neither
if not CFG["model"].startswith(MODELS[MODEL]["arch"]):
    raise RuntimeError(f"MODEL={MODEL} expects {MODELS[MODEL]['arch']}, cfg.json has {CFG['model']}")

sample_sub = pd.read_csv(DATA / "sample_submission.csv")
assert list(sample_sub.columns[1:]) == species, "species order differs from sample_submission"
print(f"model {MODEL} ({CFG['model']}, norm {CFG['norm']}) | backend {BACKEND} | workers {NUM_WORKERS} | CPUs {os.cpu_count()}")
print("Data:", DATA, "| model:", MODEL_DIR, "| Python:", sys.version.split()[0])

test_found = sorted((DATA / "test_soundscapes").glob("*.ogg"))
source = RUN_ON if RUN_ON != "auto" else ("test_soundscapes" if test_found else "train_soundscapes")
if source == "test_soundscapes":
    if not test_found:
        raise RuntimeError("RUN_ON=test_soundscapes but the folder has no .ogg files")
    files = test_found
else:
    files = sorted((DATA / "train_soundscapes").glob("*.ogg"))[:TIMING_N_FILES]
if not files:
    raise RuntimeError(f"no soundscapes found in {DATA / source}")
print(f"Running on {len(files)} files from {source}")
if source == "train_soundscapes" and len(files) < TIMING_N_FILES:
    print(f"WARNING: only {len(files)} of {TIMING_N_FILES} requested files exist")


## Worker code
We write this code to a module so that `spawn` workers can import it. The workers start clean instead of forking a process that has already run torch, ONNX Runtime or OpenVINO.

In [ ]:
LIB_SRC = r"""
import os, time
import numpy as np
import soundfile as sf
import librosa

CFG, WIN, TOP_DB, _logits, _probs, _init_s, _init_end = None, None, None, None, None, 0.0, 0.0

def configure(cfg):
    global CFG, WIN, TOP_DB
    CFG, WIN, TOP_DB = cfg, cfg["sr"] * cfg["duration"], float(cfg["norm_top_db"])

def to_input(db):
    # Same as to_input in the A1 notebooks. For norm "v2" this equals M1's (db + 80) / 80.
    db = np.maximum(db - db.max(), -TOP_DB)
    if CFG["norm"] == "zscore_minmax":
        x = (db - db.mean()) / (db.std() + 1e-6)
        return ((x - x.min()) / (x.max() - x.min() + 1e-6)).astype(np.float32)
    return ((db + TOP_DB) / TOP_DB).astype(np.float32)

def decode(path):
    y, _ = sf.read(str(path), dtype="float32")
    if y.ndim > 1:
        y = y.mean(axis=1)
    n_win = max(1, int(np.ceil(len(y) / WIN)))
    y = np.pad(y, (0, n_win * WIN - len(y)))
    return y.reshape(n_win, WIN)

def windows_to_inputs(batch):
    # One melspectrogram call for the file. Each row gets its own STFT, as in the per-window loop.
    power = librosa.feature.melspectrogram(
        y=batch, sr=CFG["sr"], n_fft=CFG["n_fft"], hop_length=CFG["hop_length"],
        n_mels=CFG["n_mels"], fmin=CFG["fmin"], fmax=CFG["fmax"])
    # ref=np.max and the normalisation stay per window.
    return np.stack([to_input(librosa.power_to_db(m, ref=np.max, top_db=TOP_DB)) for m in power])

def build_torch(cfg, n_classes, weights, exportable=False):
    import timm, torch
    kw = dict(exportable=True) if exportable else {}
    model = timm.create_model(cfg["model"], pretrained=False, in_chans=1, num_classes=n_classes, **kw)
    model.load_state_dict(torch.load(weights, map_location="cpu"))
    return model.eval()

def make_logit_fn(backend, path, cfg, n_classes):
    # Returns f(X) -> numpy logits, where X has shape (windows, mels, frames).
    if backend == "torch":
        import torch
        torch.set_num_threads(1)
        try:
            torch.set_num_interop_threads(1)
        except RuntimeError:
            pass        # already set in this process
        model = build_torch(cfg, n_classes, path)
        def run(X):
            with torch.inference_mode():
                return model(torch.from_numpy(X)[:, None]).numpy()
        return run
    if backend == "onnx":
        import onnxruntime as ort
        so = ort.SessionOptions()
        so.intra_op_num_threads = 1
        so.inter_op_num_threads = 1
        so.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
        sess = ort.InferenceSession(str(path), so, providers=["CPUExecutionProvider"])
        return lambda X: sess.run(["logits"], {"mel": X[:, None]})[0]
    if backend == "openvino_fp16":
        import openvino as ov
        # Pinning is off because several single-thread processes would otherwise compete for the same core.
        compiled = ov.Core().compile_model(str(path), "CPU", {
            "PERFORMANCE_HINT": "LATENCY", "INFERENCE_NUM_THREADS": 1, "NUM_STREAMS": 1, "ENABLE_CPU_PINNING": False})
        def run(X):
            return np.array(compiled(X[:, None])[0])
        run.compiled = compiled
        return run
    raise ValueError(backend)

def sigmoid(logits, backend):
    if backend == "torch":
        import torch
        return torch.sigmoid(torch.from_numpy(logits)).numpy()   # same call as the existing notebooks
    return (1.0 / (1.0 + np.exp(-logits.astype(np.float64)))).astype(np.float32)

def init_worker(cfg, n_classes, backend, path):
    global _logits, _probs, _init_s, _init_end
    t0 = time.perf_counter()
    configure(cfg)
    _logits = make_logit_fn(backend, path, cfg, n_classes)
    _probs = lambda l: sigmoid(l, backend)
    _init_s = time.perf_counter() - t0
    _init_end = time.time()

def predict_file(path):
    t0 = time.perf_counter()
    batch = decode(path)
    t1 = time.perf_counter()
    X = windows_to_inputs(batch)
    t2 = time.perf_counter()
    p = _probs(_logits(X)).astype(np.float32)
    t3 = time.perf_counter()
    row_ids = [f"{path.stem}_{(i + 1) * CFG['duration']}" for i in range(len(X))]
    return row_ids, p, dict(decode=t1 - t0, mel=t2 - t1, model=t3 - t2, init=_init_s, init_end=_init_end, pid=os.getpid())
"""
(WORK / "fast_infer_lib.py").write_text(LIB_SRC)
sys.path.insert(0, str(WORK.resolve()))
importlib.invalidate_caches()
import fast_infer_lib as lib
lib.configure(CFG)


## Packages
Backends other than `torch` need extra packages. We use them from the image if they are present, and otherwise install them from wheels in an attached dataset. If that fails too, the notebook stops and names what is missing.

In [ ]:
NEEDED = {"torch": [], "onnx": ["onnx", "onnxruntime"], "openvino_fp16": ["onnx", "openvino"]}[BACKEND]

def version_of(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

for name in ("torch", "timm", "librosa", "soundfile", "onnx", "onnxruntime", "openvino"):
    print(f"image package {name}: {version_of(name)}")

def wheel_dirs():
    roots = [os.environ["M3_WHEELS_DIR"]] if os.environ.get("M3_WHEELS_DIR") else ["/kaggle/input"]
    return sorted({str(Path(h).parent) for r in roots for h in glob.glob(f"{r}/**/*.whl", recursive=True)})

missing = [n for n in NEEDED if importlib.util.find_spec(n) is None]
if missing:
    dirs = wheel_dirs()
    args = [sys.executable, "-m", "pip", "install", "-q", "--no-index"]
    for d in dirs:
        args += ["--find-links", d]
    r = subprocess.run(args + missing, capture_output=True, text=True) if dirs else None
    importlib.invalidate_caches()
    still = [n for n in missing if importlib.util.find_spec(n) is None]
    if still:
        tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
        detail = r.stderr[-400:] if r is not None else "no .whl files are attached"
        raise RuntimeError(f"BACKEND={BACKEND} needs {still}, not in this image. Attach a dataset with linux x86_64 "
                           f"{tag} wheels for them and their dependencies. pip said: {detail}")
    print("Installed from attached wheels:", missing, {n: version_of(n) for n in missing})
SETUP_S = time.time() - T_NOTEBOOK


## Export and verify
We check the export against torch on the windows of one real file. The comparison uses the raw logits, so differences in the low-probability tail are not hidden by the sigmoid.

In [ ]:
EXPORT_S, VERIFY_S, VERIFY = 0.0, 0.0, None
MODEL_PATH = MODEL_DIR / "model.pth"

if BACKEND != "torch":
    import inspect
    torch.set_num_threads(1)
    t0 = time.time()
    X_check = lib.windows_to_inputs(lib.decode(files[0]))
    export_dir = WORK / "export" / MODEL
    export_dir.mkdir(parents=True, exist_ok=True)
    onnx_path = export_dir / "model.onnx"

    # Two windows here, a whole file in the check below, so the free batch axis is exercised.
    dummy = torch.from_numpy(np.repeat(X_check[:1], 2, axis=0))[:, None]
    net = lib.build_torch(CFG, len(species), MODEL_PATH, exportable=True)
    kw = {"dynamo": False} if "dynamo" in inspect.signature(torch.onnx.export).parameters else {}
    torch.onnx.export(net, (dummy,), str(onnx_path), input_names=["mel"], output_names=["logits"],
                      dynamic_axes={"mel": {0: "batch"}, "logits": {0: "batch"}}, opset_version=17,
                      do_constant_folding=True, **kw)
    MODEL_PATH = onnx_path
    if BACKEND == "openvino_fp16":
        import openvino as ov
        MODEL_PATH = export_dir / "model_fp16.xml"
        ov.save_model(ov.convert_model(str(onnx_path)), str(MODEL_PATH), compress_to_fp16=True)
    EXPORT_S = time.time() - t0

    t0 = time.time()
    tol = FP16_LOGIT_TOL if BACKEND == "openvino_fp16" else ONNX_LOGIT_TOL
    # The reference is built the way the existing inference notebooks build it (not exportable=True).
    ref = lib.make_logit_fn("torch", MODEL_DIR / "model.pth", CFG, len(species))(X_check)
    run = lib.make_logit_fn(BACKEND, MODEL_PATH, CFG, len(species))
    got = run(X_check)
    if got.shape != ref.shape:
        raise RuntimeError(f"{BACKEND} output shape {got.shape} differs from torch {ref.shape}")
    logit_diff = float(np.abs(got.astype(np.float64) - ref.astype(np.float64)).max())
    prob_diff = float(np.abs(lib.sigmoid(got, BACKEND).astype(np.float64)
                             - lib.sigmoid(ref, "torch").astype(np.float64)).max())
    VERIFY_S = time.time() - t0
    VERIFY = dict(batch=int(len(X_check)), max_abs_logit_diff=logit_diff, max_abs_prob_diff=prob_diff, logit_tolerance=tol)
    if BACKEND == "openvino_fp16":
        VERIFY["inference_precision"] = str(run.compiled.get_property("INFERENCE_PRECISION_HINT"))
    print(f"VERIFY {BACKEND} vs torch on {len(X_check)} windows | max abs logit diff {logit_diff:.3e} "
          f"(tolerance {tol:g}) | max abs prob diff {prob_diff:.3e}")
    if "inference_precision" in VERIFY:
        print("OpenVINO CPU inference precision:", VERIFY["inference_precision"])
    if not np.isfinite(logit_diff) or logit_diff > tol:
        raise RuntimeError(f"{BACKEND} export differs from torch: {logit_diff} > {tol}")
    del run, ref, got, net
    print(f"Export {EXPORT_S:.1f}s | verify {VERIFY_S:.1f}s | artefact {MODEL_PATH}")


## Predict

In [ ]:
initargs = (CFG, len(species), BACKEND, MODEL_PATH)
PIPELINE_T0 = time.time()
if NUM_WORKERS > 0:
    ctx = mp.get_context("spawn")
    with ctx.Pool(NUM_WORKERS, initializer=lib.init_worker, initargs=initargs) as pool:
        results = list(pool.imap(lib.predict_file, files, chunksize=1))
else:
    lib.init_worker(*initargs)
    results = [lib.predict_file(p) for p in files]
PIPELINE_WALL_S = time.time() - PIPELINE_T0

row_ids = [r for ids, _, _ in results for r in ids]
preds = np.concatenate([p for _, p, _ in results])
print("Predictions:", preds.shape, f"in {PIPELINE_WALL_S:.1f}s")


## Timing
`timing_profile.json` uses the same keys as the profiling notebook. `decode`, `mel` and `model_forward` are summed over workers (compute seconds), so `per_file_mean_s` can be compared with a 1-worker profile. The elapsed figures are `preprocess_elapsed` (an estimate), `total_pipeline` and the projection.

In [ ]:
n = len(files)
stat = [r[2] for r in results]
stage = lambda k: sum(s[k] for s in stat)
workers = {s["pid"]: s for s in stat}
n_active = len(workers)
DECODE_S, MEL_S, MODEL_FORWARD_S = stage("decode"), stage("mel"), stage("model")
MODEL_LOAD_S = max(s["init"] for s in workers.values())
# Elapsed time until the slowest reporting worker was ready (spawn, imports, model load).
STARTUP_WALL_S = min(max(max(s["init_end"] for s in workers.values()) - PIPELINE_T0, 0.0), PIPELINE_WALL_S)
# Parallel work cannot beat the summed compute time divided by the workers, which bounds short runs.
per_file_wall = max((PIPELINE_WALL_S - STARTUP_WALL_S) / n, (DECODE_S + MEL_S + MODEL_FORWARD_S) / n_active / n)
fixed_s = SETUP_S + EXPORT_S + VERIFY_S + STARTUP_WALL_S
projected_min = (fixed_s + per_file_wall * 700) / 60

stages = {
    "setup": SETUP_S,
    "export": EXPORT_S,
    "verify": VERIFY_S,
    "model_load": MODEL_LOAD_S,
    "decode": DECODE_S,
    "mel": MEL_S,
    "preprocess_elapsed": (DECODE_S + MEL_S) / n_active,
    "model_forward": MODEL_FORWARD_S,
    "worker_startup_elapsed": STARTUP_WALL_S,
    "total_pipeline": per_file_wall * n,
    "total_notebook": time.time() - T_NOTEBOOK,
}
per_file = ["decode", "mel", "preprocess_elapsed", "model_forward", "total_pipeline"]
profile = {
    "n_files_requested": TIMING_N_FILES if source == "train_soundscapes" else n,
    "n_files_used": n,
    "n_windows": len(preds),
    "num_workers": NUM_WORKERS,
    "torch_threads": 1,
    "cpu_count": os.cpu_count(),
    "batch_size": int(max(len(p) for _, p, _ in results)),
    "stages_s": {k: round(v, 4) for k, v in stages.items()},
    "per_file_mean_s": {k: round(stages[k] / n, 6) for k in per_file},
    "device": DEVICE,
    "cuda_available": CUDA_AVAILABLE,
    "model": MODEL,
    "model_arch": CFG["model"],
    "backend": BACKEND,
    "source": source,
    "active_workers": n_active,
    "per_file_elapsed_s": round(per_file_wall, 6),
    "projected_700_files_min": round(projected_min, 2),
    "budget_min": BUDGET_MIN,
    "verify": VERIFY,
    "timing_note": "decode, mel, model_forward are summed over workers; preprocess_elapsed is their sum divided by active workers",
}
with open(WORK / "timing_profile.json", "w") as f:
    json.dump(profile, f, indent=2)
print(f"TIMING | decode {DECODE_S:.1f}s | mel {MEL_S:.1f}s | model {MODEL_FORWARD_S:.1f}s (summed over {n_active} workers)")
print(f"TIMING | setup {SETUP_S:.1f}s | export {EXPORT_S:.1f}s | verify {VERIFY_S:.1f}s | worker startup {STARTUP_WALL_S:.1f}s "
      f"(model load {MODEL_LOAD_S:.1f}s) | {per_file_wall:.2f}s per file elapsed")
print(f"TIMING | projected 700 files: {projected_min:.1f} min (budget {BUDGET_MIN}) -> "
      f"{'OK' if projected_min <= BUDGET_MIN else 'OVER BUDGET'} | measured on {n} files")


In [ ]:
sub = pd.DataFrame(preds, columns=species)
sub.insert(0, "row_id", row_ids)
sub.to_csv(WORK / "submission.csv", index=False)
print(sub.shape)
if source != "test_soundscapes":
    print("NOTE: submission.csv covers training soundscapes; this was a timing run.")
sub.iloc[:3, :4]
